# Stage 2 Notebook 16 - Exp2K CLRKD lane head + LineIoU regression cls target

Exp2G/H/I/J all converged on the same broken cls equilibrium: across architectures (single path vs separate path), losses (focal vs ASL), and sampling (OHEM vs all priors), the cls head ended at val/lane_exist_best_f1 ~ 0.05-0.08 with pos and neg both stuck near sigmoid(logit)=0.57 (no separation). The Exp2I e2 spike of 0.535 was a one-shot optimizer artifact that none of the subsequent ablations reproduced.

Diagnosis after four architecture-side attempts: the binary existence target itself is the bottleneck. Dynamic-k matching makes the same prior positive in some batches and negative in others (the matching outcome depends on which competing priors win the IoU contest), so the cls head sees contradictory signals across batches and converges to a degenerate 'predict 0.5 for everything' equilibrium. No feature/loss/optimizer fix can resolve a target that flips on its own.

Exp2K removes the matching dependency from the cls supervision entirely:

- Replace binary {0,1} cls target with **continuous LineIoU regression**. For each prior, target = max LineIoU between this prior's predicted curve (detached) and any valid GT lane in the same image. Plain BCE-with-logits on the continuous target.
- This target is **deterministic per batch** (computed from the model's current geometry, not from a competitive matching outcome), so the cls head no longer sees contradictory supervision for the same prior across different batches.
- The semantics also match what we actually want at inference: a per-prior IoU score for top-K decode + lane NMS (planned next step). CLRNet and RTMDet both use IoU-aware quality scores.

Removed confounds (one-goal-at-a-time):

- **OHEM disabled** (`cls_ohem_topk_per_pos: 0`) — continuous target is balanced; no class-imbalance issue.
- **ASL disabled** (`cls_loss_type: focal` is unused under regression). BCE works directly on a continuous target.
- **Separate cls path disabled** (`cls_separate_path: false`) — Exp2J showed it hurts geometry without helping cls.
- Single shared aggregator (Exp2I-style), `w_iou: 2.0` (Exp2G's geometry weight, healthy).

Kept: ROI gather + multi-scale + 3-stage refinement + dynamic-k matching for **geometry losses only** + prior_embed_encoder + RMT+GCA backbone.

Hypothesis: with a stable per-prior target, the cls head should learn meaningful separation. If it does, val/lane_exist_best_f1 should rise above 0.30 by epoch 10 (a 5x jump from the Exp2G/H/I/J cluster), and val/clrkd_style_f1 should rise from ~0.02.

Reference: RTMDet 'Quality Focal Loss' formulation; CLRNet quality-aware label assignment.

### Run mode

1. Keep `DEBUG_MODE = True` for the first run.
2. After the smoke and debug run succeed, change to `False` for the 10-epoch short run.
3. Output is mirrored to the notebook cell, the Colab runtime log, and a Drive log file.
4. Do not rerun Notebook 00.

In [1]:
import os, sys, subprocess, textwrap
from google.colab import drive
os.environ['PYTHONUNBUFFERED'] = '1'
drive.mount('/content/drive')

REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
if not os.path.isdir(REPO_ROOT):
    raise FileNotFoundError(f'Missing project root: {REPO_ROOT}')
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'scipy', 'opencv-python-headless', 'tqdm', 'matplotlib'])
print('repo:', REPO_ROOT)

from stage2.scripts.notebook_utils import run_streaming
LOG_DIR = '/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs'
os.makedirs(LOG_DIR, exist_ok=True)

Mounted at /content/drive
repo: /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane


In [2]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp11_rmt_gca_clrkd_iou_regression_joint.yaml'
LOG_FILE = os.path.join(LOG_DIR, f'{Path(CONFIG).stem}_smoke.log')

# Smoke test: forward + backward through the IoU-regression cls path.
# Must print 'OK exp11_*.yaml' with shapes lane_shape=(1, 16, 72, 2)
# det_shape=(1, 4, 4) before training is attempted.
run_streaming([sys.executable, '-u', 'stage2/scripts/smoke_test_joint_models.py', CONFIG], log_path=LOG_FILE)

[run_streaming] command: /usr/bin/python3 -u stage2/scripts/smoke_test_joint_models.py stage2/configs/exp11_rmt_gca_clrkd_iou_regression_joint.yaml
[run_streaming] log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp11_rmt_gca_clrkd_iou_regression_joint_smoke.log
OK exp11_rmt_gca_clrkd_iou_regression_joint.yaml
  lane_shape=(1, 16, 72, 2) det_shape=(1, 4, 4)
  lane_loss=11.4504 det_loss=3.6140 grad_cos=-0.0688 lambda_lane=0.0500
  gate_stats={'gate/det_mean': 0.500764787197113, 'gate/lane_mean': 0.4987189471721649, 'gate/det_sat_low': 0.0, 'gate/det_sat_high': 0.0, 'gate/lane_sat_low': 0.0, 'gate/lane_sat_high': 0.0}
[run_streaming] return_code=0


0

In [3]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp11_rmt_gca_clrkd_iou_regression_joint.yaml'
CURVE_TAR = '/content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar'
CURVE_ROOT = '/content/bdd100k_clrkd_curve'

DEBUG_MODE = False

if DEBUG_MODE:
    RUN_TAG = 'debug'
    EPOCHS = 2
    BATCH_SIZE = 4
    LIMIT_TRAIN = 512
    LIMIT_VAL = 256
    PRINT_EVERY = 5
else:
    RUN_TAG = 'short10'
    EPOCHS = 10
    BATCH_SIZE = 8
    LIMIT_TRAIN = 3000
    LIMIT_VAL = 1000
    PRINT_EVERY = 5

run_stem = Path(CONFIG).stem + '_' + RUN_TAG
WORK_DIR = f'/content/{run_stem}'
OUTPUT_TAR = f'/content/drive/MyDrive/EcoCAR/training_runs/{run_stem}.tar'
LOG_FILE = os.path.join(LOG_DIR, f'{run_stem}_train.log')

cmd = [
    sys.executable, '-u', 'stage2/scripts/train_joint_model_experiment.py',
    '--config', CONFIG,
    '--curve-tar', CURVE_TAR,
    '--curve-root', CURVE_ROOT,
    '--work-dir', WORK_DIR,
    '--output-tar', OUTPUT_TAR,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--limit-train', str(LIMIT_TRAIN),
    '--limit-val', str(LIMIT_VAL),
    '--force-extract',
    '--print-every', str(PRINT_EVERY),
]

print('DEBUG_MODE:', DEBUG_MODE, flush=True)
print('About to run:', ' '.join(cmd), flush=True)
print('Output tar:', OUTPUT_TAR, flush=True)
print('Visible log file:', LOG_FILE, flush=True)
run_streaming(cmd, log_path=LOG_FILE)

DEBUG_MODE: False
About to run: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp11_rmt_gca_clrkd_iou_regression_joint.yaml --curve-tar /content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar --curve-root /content/bdd100k_clrkd_curve --work-dir /content/exp11_rmt_gca_clrkd_iou_regression_joint_short10 --output-tar /content/drive/MyDrive/EcoCAR/training_runs/exp11_rmt_gca_clrkd_iou_regression_joint_short10.tar --epochs 10 --batch-size 8 --limit-train 3000 --limit-val 1000 --force-extract --print-every 5
Output tar: /content/drive/MyDrive/EcoCAR/training_runs/exp11_rmt_gca_clrkd_iou_regression_joint_short10.tar
Visible log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp11_rmt_gca_clrkd_iou_regression_joint_short10_train.log
[run_streaming] command: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp11_rmt_gca_clrkd_iou_regression_joint.yaml --curve-tar /content/drive/MyDrive

0

## What to watch in Exp2K training

Reference epoch 10 of the matched-existence experiments:
- Exp2G: best_f1=0.075, pos-neg=+0.004, point_mae=0.324, matched_iou=0.428.
- Exp2H: best_f1=0.083, pos-neg=+0.016, point_mae=0.329, matched_iou=0.402.
- Exp2I: best_f1=0.057, pos-neg=+0.001, point_mae=0.329, matched_iou=0.391.
- Exp2J: best_f1=0.053, pos-neg=+0.000, point_mae=0.334, matched_iou=0.362.

Strong signals that the IoU regression target unlocked cls:

- `val/lane_exist_best_f1` >= 0.30 by epoch 5 and >= 0.50 by epoch 10. Most importantly: it rises *monotonically* (the four prior experiments had only transient peaks).
- `val/lane_exist_pos_score_mean - val/lane_exist_neg_score_mean` >= 0.20 at epoch 10. The score now means 'predicted IoU', so matched priors (high IoU) should be far above unmatched priors (low IoU).
- `val/lane/cls_pos` and `val/lane/cls_neg` are both BCE losses on a continuous target. Their absolute values are no longer directly comparable to focal/ASL runs. What matters: both should *decrease* over training, with cls_pos lower than cls_neg (matched priors carry higher IoU targets, so the model is more confident on them).
- `val/lane/clrkd_style_f1` rises above 0.05 (vs the 0.018-0.022 cluster in prior experiments). This is the project's actual end-task metric closest to CLRKDNet's reported numbers.
- `pred_lanes / batch` drops below 500 (was stuck at ~1535 across G/H/I/J because both pos and neg were at 0.57, all above thr=0.3).
- **Geometry holds**: `val/lane_point_mae` <= 0.34 and `val/matched_line_iou` >= 0.40 at epoch 10 (close to or matching Exp2G's 0.428).

Failure signals:

- best_f1 below 0.20 at epoch 10 -> the IoU target itself is the wrong signal for this dataset's prior layout, or the head architecture cannot represent it. Next: compute the IoU target against per-prior nearest-GT only (rather than max over all GTs) so the target is more local and discriminative.
- `pred_lanes / batch` collapses to near zero -> the IoU target distribution is too low (most priors get target ~0). Fix: rescale the target via `target = clamp(2 * iou - 0.5, 0, 1)` to spread it out.
- Geometry regresses (point_mae > 0.36) -> ASL/OHEM removal hurt; reintroduce focal on the IoU target.

After short10 finishes, run Notebook 08 (now includes exp11 candidates) to plot Exp2G / Exp2H / Exp2I / Exp2J / Exp2K side-by-side.

If Exp2K passes, the next logical step is **lane decode + NMS** so val/clrkd_style_f1 reflects proper top-K predictions vs GT, which is the metric we need to compare to CLRKDNet.